# Validation 2: Sarvam vs OpenAI Speech-to-Text

**Objective:** Compare **Sarvam saaras:v3** (5 STT modes) against **OpenAI** (`gpt-4o-mini-transcribe` / `whisper-1`) on custom recordings — codemixed Hindi–English, pure Hindi, silence, short clips, and the 30-second API limit.

1. Run cells **3 → 7** once (setup → STT providers → Gradio viewer)
2. Run any recording section below

**Output:** Gradio panel with **input audio** (left) and **transcripts** (right) — all Sarvam modes stacked + one OpenAI result.<br><br>
**Sarvam modes:** transcribe · translate · verbatim · translit · codemix<br>
**OpenAI:** set `TRANSLATE = True` for whisper-1 English translation, `False` for transcription only.<br><br>
All recordings live in `test_inputs/speech recordings/`.

## Summary of results: <br>

| Sr no  | File type  | Description of input | What it tests? | Transcription result (Sarvam)  | Transcription result (OpenAI with translate=False) | Runtime | Comments
|---|---|---|---|---|---|---|--|
|1 | STT_codemixed.mp3 | Code-mixed language with numbers [108 chars]| Code-mixed language handling, number preservation | Preserves meaning in translation but no code-mixed mode present | 100% accurate across all 5 modes | [Sarvam 5 modes] 4.58s <br> [OpenAI -> English] 3.34s| |
|2| STT_english_31seconds.mp3 | Edge case 31s audio| Error handling | Pass | Correct Error code, REST API can only have audio upto 30s| NA | | 
|3 |STT_pure_hindi.mp3 | Pure hindi language handling [102 chars] | Preserves meaning | Accurate and no difference in the 5 mode output | [Sarvam all 5 modes] 9.96s [OpenAI incl translation] 1.41s
|4 |STT_silence.mp3 | Background white noise and no speech [0 chars]| Voice detection precision | Empty as expected| No hallucination; empty as expected |[Sarvam 5 modes] 7.06s [OpenAI ] 0.97s| Need to check no character recordings |
|5 |STT_2seconds.mp3 | Very short utterance (2 chars) | Processing single utterrances | Pass | Pass | [Sarvam total 5 modes] 2.06s [OpenAI] 1.67s| |
|6 | STTsample_product_refund.mp3 | Customer refund clip | Longer real-world utterance | | | | |

## Setup — paths, API key, audio corpus

In [ ]:
import shutil
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
from sarvamai import SarvamAI

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")

# audio folder name has spaces — we cache copies for Gradio playback (see _gradio_audio_path)
AUDIO_DIR = ROOT / "test_inputs/speech recordings"
RESULTS_DIR = ROOT / "results"
GRADIO_AUDIO_CACHE = RESULTS_DIR / "_gradio_audio_cache"
RESULTS_DIR.mkdir(exist_ok=True)
GRADIO_AUDIO_CACHE.mkdir(exist_ok=True)

# all 5 Sarvam saaras modes we run per file
STT_MODES: list[str] = [
    "transcribe",
    "translate",
    "verbatim",
    "translit",
    "codemix",
]

_client: SarvamAI | None = None
_openai_client: OpenAI | None = None


def get_sarvam_client() -> SarvamAI:
    """Reuse one Sarvam client — avoids re-auth on every mode call."""
    global _client
    if _client is None:
        import os
        api_key = os.getenv("SARVAM_API_KEY")
        if not api_key:
            raise ValueError("SARVAM_API_KEY not set — add it to .env")
        _client = SarvamAI(api_subscription_key=api_key)
    return _client


def get_openai_client() -> OpenAI:
    """OpenAI client for gpt-4o-mini-transcribe / whisper-1."""
    global _openai_client
    if _openai_client is None:
        _openai_client = OpenAI()
    return _openai_client


print("AUDIO_DIR exists:", AUDIO_DIR.exists())
print("Recordings:", sorted(p.name for p in AUDIO_DIR.glob("*.mp3")))

AUDIO_DIR exists: True
Recordings: ['STT_2seconds_empty.mp3', 'STT_codemixed.mp3', 'STT_english_31seconds_edge_case.mp3', 'STT_natural_pure_hindi.mp3', 'STT_silence.mp3', 'STTsample_product_refund.mp3']


## STT providers — Sarvam saaras:v3 + OpenAI

In [ ]:
@dataclass
class SttResult:
    """One audio file × all Sarvam saaras:v3 mode outputs."""
    audio_path: Path
    success: bool
    runtime_total_seconds: float
    model: str = "saaras:v3"
    transcripts: dict[str, str] = field(default_factory=dict)
    raw_responses: dict[str, Any] = field(default_factory=dict)
    errors: dict[str, str] = field(default_factory=dict)
    error_type: str | None = None
    error_message: str | None = None
    error_code: str | int | None = None

    @property
    def name(self) -> str:
        return self.audio_path.name


@dataclass
class OpenAiSttResult:
    """Single OpenAI transcription or translation result."""
    audio_path: Path
    success: bool
    runtime_total_seconds: float
    translate: bool
    model: str
    text: str = ""
    raw_response: dict[str, Any] | None = None
    error_type: str | None = None
    error_message: str | None = None
    error_code: str | int | None = None

    @property
    def name(self) -> str:
        return self.audio_path.name

    @property
    def task_label(self) -> str:
        return "translation (→ English)" if self.translate else "transcription"


def _extract_error_info(exc: Exception) -> tuple[str, str, str | int | None]:
    from sarvamai.core.api_error import ApiError
    error_type = type(exc).__name__
    error_message = str(exc)
    error_code: str | int | None = None
    if isinstance(exc, ApiError):
        error_code = exc.status_code
        if exc.body is not None:
            error_message = str(exc.body)
    elif getattr(exc, "status_code", None) is not None:
        error_code = getattr(exc, "status_code")
    return error_type, error_message, error_code


def run_sarvam_stt(audio_path: str | Path, *, model: str = "saaras:v3", modes: list[str] | None = None) -> SttResult:
    """Run every Sarvam mode on one file; per-mode errors stored in transcripts dict."""
    audio_path = Path(audio_path)
    modes = list(modes or STT_MODES)
    started = time.perf_counter()
    if not audio_path.exists():
        exc = FileNotFoundError(audio_path)
        rt = round(time.perf_counter() - started, 2)
        et, em, ec = _extract_error_info(exc)
        return SttResult(audio_path=audio_path, success=False, runtime_total_seconds=rt, error_type=et, error_message=em, error_code=ec)
    client = get_sarvam_client()
    transcripts, raw_responses, errors = {}, {}, {}
    print(f"[Sarvam] {audio_path.name} ({len(modes)} modes)...")
    for mode in modes:
        t0 = time.perf_counter()
        try:
            with open(audio_path, "rb") as f:
                resp = client.speech_to_text.transcribe(file=f, model=model, mode=mode)
            transcripts[mode] = resp.transcript or ""
            raw_responses[mode] = resp.model_dump()
            print(f"  ✓ {mode:12} ({round(time.perf_counter()-t0,2)}s)")
        except Exception as exc:
            et, em, ec = _extract_error_info(exc)
            errors[mode] = em
            transcripts[mode] = f"**Error** ({et}): {em}"
            print(f"  ✗ {mode:12} — {em}")
    rt = round(time.perf_counter() - started, 2)
    print(f"[Sarvam] total {rt}s")
    return SttResult(audio_path=audio_path.resolve(), success=len(errors) < len(modes), runtime_total_seconds=rt, model=model, transcripts=transcripts, raw_responses=raw_responses, errors=errors)


def run_openai_stt(audio_path: str | Path, *, translate: bool = False, transcribe_model: str = "gpt-4o-mini-transcribe", translate_model: str = "whisper-1") -> OpenAiSttResult:
    """One OpenAI call — transcription (gpt-4o-mini) or translation (whisper-1)."""
    audio_path = Path(audio_path)
    started = time.perf_counter()
    model = translate_model if translate else transcribe_model
    task = "translation" if translate else "transcription"
    if not audio_path.exists():
        exc = FileNotFoundError(audio_path)
        rt = round(time.perf_counter() - started, 2)
        et, em, ec = _extract_error_info(exc)
        return OpenAiSttResult(audio_path=audio_path, success=False, runtime_total_seconds=rt, translate=translate, model=model, error_type=et, error_message=em, error_code=ec)
    client = get_openai_client()
    print(f"[OpenAI] {audio_path.name} — {task} ({model})...")
    try:
        with open(audio_path, "rb") as f:
            if translate:
                resp = client.audio.translations.create(model=translate_model, file=f)
            else:
                resp = client.audio.transcriptions.create(model=transcribe_model, file=f)
        text = resp.text or ""
        rt = round(time.perf_counter() - started, 2)
        print(f"[OpenAI] ✓ {rt}s — {len(text)} chars")
        return OpenAiSttResult(audio_path=audio_path.resolve(), success=True, runtime_total_seconds=rt, translate=translate, model=model, text=text, raw_response=resp.model_dump())
    except Exception as exc:
        rt = round(time.perf_counter() - started, 2)
        et, em, ec = _extract_error_info(exc)
        print(f"[OpenAI] ✗ {et} (code {ec}): {em}")
        return OpenAiSttResult(audio_path=audio_path.resolve(), success=False, runtime_total_seconds=rt, translate=translate, model=model, text=f"**Error**: {em}", error_type=et, error_message=em, error_code=ec)


def run_stt_comparison(audio_path: str | Path, *, translate: bool = False) -> tuple[SttResult, OpenAiSttResult]:
    """Convenience wrapper — Sarvam all modes + one OpenAI pass."""
    return run_sarvam_stt(audio_path), run_openai_stt(audio_path, translate=translate)

## Gradio viewer — audio + side-by-side transcripts

In [ ]:
def _format_all_transcripts(sarvam: SttResult, openai: OpenAiSttResult | None = None) -> str:
    """Markdown blob for the right panel — all Sarvam modes + optional OpenAI block."""
    parts = ["## Sarvam (saaras:v3)"]
    for mode in STT_MODES:
        parts.append(f"### {mode}\n\n{sarvam.transcripts.get(mode, '_(not run)_')}")
    if openai:
        parts += ["---", f"## OpenAI — {openai.task_label}", f"_{openai.model} · {openai.runtime_total_seconds}s_", openai.text or "_(empty)_"]
    return "\n\n".join(parts)


def _gradio_audio_path(audio_path: Path) -> str:
    """Copy audio into cache so Gradio can serve it (folder name has spaces)."""
    cached = GRADIO_AUDIO_CACHE / audio_path.name
    if not cached.exists() or cached.stat().st_mtime < audio_path.stat().st_mtime:
        shutil.copy2(audio_path, cached)
    return str(cached.resolve())


def show_stt_result(sarvam: SttResult, openai: OpenAiSttResult | None = None, *, window_height: int = 1100) -> gr.Blocks:
    """Gradio: cached audio (left) | transcripts (right)."""
    audio_path = _gradio_audio_path(sarvam.audio_path)
    status = f"**Sarvam** {sarvam.runtime_total_seconds}s"
    if openai:
        status += f" · **OpenAI** {openai.runtime_total_seconds}s ({openai.task_label})"
    with gr.Blocks(title=sarvam.name) as demo:
        gr.Markdown(f"### {sarvam.name}\n\n{status}")
        with gr.Row(equal_height=True):
            gr.Audio(
                value=audio_path,
                label="Input audio",
                type="filepath",
                sources=[],
                interactive=False,
                show_download_button=True,
            )
            gr.Markdown(value=_format_all_transcripts(sarvam, openai), label="Transcripts")
    demo.launch(
        inline=True,
        height=window_height,
        share=False,
        quiet=True,
        allowed_paths=[str(ROOT), str(AUDIO_DIR), str(RESULTS_DIR), str(GRADIO_AUDIO_CACHE)],
    )
    return demo

## STT_codemixed.mp3 — code-mixed speech + numbers

Tests number preservation and codemixed handling. `TRANSLATE=True` → OpenAI whisper-1 to English.

In [ ]:
TRANSLATE = True  # True → OpenAI whisper-1 English translation
stt_codemixed, openai_codemixed = run_stt_comparison(AUDIO_DIR / "STT_codemixed.mp3", translate=TRANSLATE)
show_stt_result(stt_codemixed, openai_codemixed)

[Sarvam] STT_codemixed.mp3 (5 modes)...
  ✓ transcribe   (1.25s)
  ✓ translate    (0.83s)
  ✓ verbatim     (1.69s)
  ✓ translit     (1.19s)
  ✓ codemix      (1.0s)
[Sarvam] total 5.95s
[OpenAI] STT_codemixed.mp3 — translation (whisper-1)...
[OpenAI] ✓ 2.36s — 108 chars


Gradio Blocks instance: 0 backend functions
-------------------------------------------

## STT_english_31seconds_edge_case.mp3 — 31s audio (>30s REST limit)

Expect Sarvam to reject all 5 modes; OpenAI should still transcribe.

In [ ]:
TRANSLATE = False
stt_english_31seconds_edge_case, openai_english_31seconds_edge_case = run_stt_comparison(AUDIO_DIR / "STT_english_31seconds_edge_case.mp3", translate=TRANSLATE)
show_stt_result(stt_english_31seconds_edge_case, openai_english_31seconds_edge_case)

[Sarvam] STT_english_31seconds_edge_case.mp3 (5 modes)...
  ✗ transcribe   — {'error': {'message': 'Audio duration exceeds the maximum limit of 30 seconds. Please use the batch API for longer audio files.', 'code': 'invalid_request_error', 'request_id': '20260520_ef1303ef-7c1e-4888-b8de-dc0867b33c11'}}
  ✗ translate    — {'error': {'message': 'Audio duration exceeds the maximum limit of 30 seconds. Please use the batch API for longer audio files.', 'code': 'invalid_request_error', 'request_id': '20260520_9c4d493a-878d-4be2-b909-fdb1fe138ae0'}}
  ✗ verbatim     — {'error': {'message': 'Audio duration exceeds the maximum limit of 30 seconds. Please use the batch API for longer audio files.', 'code': 'invalid_request_error', 'request_id': '20260520_7c01872f-b6cc-43d3-8184-922e34145b7e'}}
  ✗ translit     — {'error': {'message': 'Audio duration exceeds the maximum limit of 30 seconds. Please use the batch API for longer audio files.', 'code': 'invalid_request_error', 'request_id': '2026052

Gradio Blocks instance: 0 backend functions
-------------------------------------------

## STT_natural_pure_hindi.mp3 — natural pure Hindi

Check whether all 5 Sarvam modes agree; compare OpenAI translation latency.

In [ ]:
TRANSLATE = True  # whisper-1 → English
stt_natural_pure_hindi, openai_natural_pure_hindi = run_stt_comparison(AUDIO_DIR / "STT_natural_pure_hindi.mp3", translate=TRANSLATE)
show_stt_result(stt_natural_pure_hindi, openai_natural_pure_hindi)

[Sarvam] STT_natural_pure_hindi.mp3 (5 modes)...
  ✓ transcribe   (2.5s)
  ✓ translate    (2.0s)
  ✓ verbatim     (1.77s)
  ✓ translit     (1.92s)
  ✓ codemix      (1.78s)
[Sarvam] total 9.96s
[OpenAI] STT_natural_pure_hindi.mp3 — translation (whisper-1)...
[OpenAI] ✓ 1.41s — 102 chars


Gradio Blocks instance: 0 backend functions
-------------------------------------------

## STT_silence.mp3 — white noise, no speech

Voice-activity / hallucination check — both providers should return empty or near-empty.

In [ ]:
TRANSLATE = False
stt_silence, openai_silence = run_stt_comparison(AUDIO_DIR / "STT_silence.mp3", translate=TRANSLATE)
show_stt_result(stt_silence, openai_silence)

[Sarvam] STT_silence.mp3 (5 modes)...
  ✓ transcribe   (1.69s)
  ✓ translate    (1.54s)
  ✓ verbatim     (1.5s)
  ✓ translit     (1.5s)
  ✓ codemix      (1.43s)
[Sarvam] total 7.66s
[OpenAI] STT_silence.mp3 — transcription (gpt-4o-mini-transcribe)...
[OpenAI] ✓ 0.78s — 0 chars


Gradio Blocks instance: 0 backend functions
-------------------------------------------

## STT_2seconds_empty.mp3 — very short utterance (~2 chars)

Minimal audio length handling.

In [ ]:
TRANSLATE = False
stt_empty, openai_empty = run_stt_comparison(AUDIO_DIR / "STT_2seconds_empty.mp3", translate=TRANSLATE)
show_stt_result(stt_empty, openai_empty)

[Sarvam] STT_2seconds_empty.mp3 (5 modes)...
  ✓ transcribe   (0.66s)
  ✓ translate    (0.32s)
  ✓ verbatim     (0.27s)
  ✓ translit     (0.47s)
  ✓ codemix      (0.34s)
[Sarvam] total 2.06s
[OpenAI] STT_2seconds_empty.mp3 — transcription (gpt-4o-mini-transcribe)...
[OpenAI] ✓ 1.67s — 4 chars


Gradio Blocks instance: 0 backend functions
-------------------------------------------

## STTsample_product_refund.mp3 — product refund sample

Longer real-world customer-service clip.

In [ ]:
TRANSLATE = False
stt_sample_product_refund, openai_sample_product_refund = run_stt_comparison(
    AUDIO_DIR / "STTsample_product_refund.mp3", translate=TRANSLATE
)
show_stt_result(stt_sample_product_refund, openai_sample_product_refund)